# 双窗口特征重要性对比 — Edina figure guide 第 0–3 步

接在 `run_improved_pipeline_v4.ipynb` 之后,回答的是**另一个问题**:

> v4 问的是"哪个窗口最能预测结局"(答案:晚期)。
> 这里问的是"**给定两个固定窗口,重要特征集合变了多少**"——这才是 guide 的核心。

对应 guide 的章节:

| 本 notebook | guide | 内容 |
|---|---|---|
| 第 0 步 | §0 | 两个固定窗口,轨迹级平均,模型/split/seed 全程一致 |
| 第 1 步 | §1 | 真实特征名(224 残基 + 50 个按 rank 定义的水) |
| 第 2 步 | §2 | 三种重要性:Gini / permutation / SHAP |
| 第 3 步 | §3 | 方法一致性 Spearman + top-15 Jaccard → Fig 1 |
| 第 3b 步 | — | 窗口内 FI 复现基线 |
| 第 3c 步 | — | split-half 地板 + 衰减校正 |

**3b / 3c 是 guide 里没有、但必须做的一步。** Edina 的 headline 是跨窗口 ρ ≈ 0.05
(guide §8),但 Ayidh 在群里说过 "the top FIs are not the same for me from one model
to another" —— FI 本身就不稳。n=173、每折测试集才 ~35 条,permutation importance
完全可能跟**任何东西**都不相关。所以 ρ≈0 有两种解释:

- **(a)** 两个 regime 真的由不同坐标主导 ← 想要的结论
- **(b)** FI 噪声太大,ρ 对什么都是 0 ← 必须先排除

区分办法:先量出 FI 自己和自己的相关性(地板),再看跨窗口是否**明显低于**地板。
第 3c 步会直接给出一句可用的结论。

---

### 怎么跑

1. 改**第 0 步**的四个路径,其余不用动
2. `SMOKE_TEST = True` 先跑一遍(几分钟),确认路径能读、残基名对得上
3. 改成 `SMOKE_TEST = False` 跑正式的(十几分钟到半小时)
4. 从上到下顺序执行。第 2 步跑完会自动存盘,之后重启 kernel 也能从缓存继续

> **不要跳着跑。** Pedro 审计 Ayidh 时点名的第 3 条问题就是
> "notebook 的源码和保存的输出对不上,说明没有干净地从上到下跑过"。

In [1]:
# ===== 依赖 =====
# numpy / scipy / scikit-learn / h5py  —— 必需
# mdtraj  —— 特征命名用;缺了会退回 CV_i 并大声警告(那样出不了最终图)
# shap    —— 缺了自动跳过,其余照跑
#
# guide 提过 shap 和环境里的 numpy-1 冲突,隔离安装:
#   pip install --target ~/shap_env shap
#   PYTHONPATH=~/shap_env jupyter notebook
import json
import os
import time
import warnings
from itertools import combinations
from typing import Dict, List, Optional, Tuple

import h5py
import numpy as np
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold

print("numpy", np.__version__)
try:
    import mdtraj as md
    # 将 md.version.version 替换为 md.__version__
    print("mdtraj", md.__version__)
    HAVE_MDTRAJ = True
except ImportError:
    print("mdtraj 不可用 —— 特征名会退回 CV_i")
    HAVE_MDTRAJ = False
try:
    import shap
    print("shap", shap.__version__)
    HAVE_SHAP = True
except ImportError:
    print("shap 不可用 —— 会跳过 SHAP,只跑 Gini + permutation")
    HAVE_SHAP = False

numpy 2.2.6
mdtraj 1.10.3
shap 不可用 —— 会跳过 SHAP,只跑 Gini + permutation


## 第 0 步 — 配置

guide §0 的硬要求:两个窗口之间**唯一变化的只能是窗口**。所以模型超参、折数、
seed 列表在这里定死一份,两个窗口共用。

**不要给某个窗口单独调参** —— 那样模型复杂度就成了第二个变量,后面 FI 的差异
就说不清是窗口造成的还是模型造成的。

### 两个刻意的选择

**模型换成 RF,不用 v4 的 GBDT。** v4 是 `max_depth=1` × 150 棵树,274 个特征里
最多只用得到 150 个,剩下的 Gini 恒等于 0、SHAP 跟着退化,第 3 步的三方法对比
会失真。而且 guide §8 的 demo 和 Edina 群里那张复现 Ayidh 的表都是 RF,用 RF 才
可比。GBDT 留给 v4 原来的预测性能结论。

**10 个 seed,不是 5 个。** guide §0 只要求 ≥5,但第 3c 步的 split-half 地板需要
偶数个 seed 拆成 5v5,才能和"5-seed 平均"的 headline 降噪程度匹配。用 5 个拆 2v3
会让地板偏低、结论虚高。代价是 2 倍算力。

### late 窗口的一个出入

guide §0 写的是 `(2000, 2499)`,但 Edina 群里那张复现 Ayidh 模型的表用的是
`(2100, 2499)`。主图用 guide 的版本,`(2100, 2499)` 作敏感性检查一起跑,报告里
交代一句就行。

In [2]:
# ============================ 路径 ============================
H5_PATH     = "/mnt/data1/student/trypsin/h5_datasets/CV_database_4864_TS_variedDCDfreq_lmin2p6_lmax4p0.h5"
PDB_PATH    = "/mnt/data1/student/trypsin/h5_datasets/step3_input.pdb"
WATERS_PATH = "/mnt/data1/student/trypsin/h5_datasets/4864_TS_50closest_waters.npy"
OUT_DIR     = "/mnt/data1/student/trypsin/yucheng/07.12/v4"

# ============================ 开关 ============================
# 先用 True 跑一遍确认路径/命名/落盘都通,再改 False 跑正式的。
# 冒烟模式用 6 个 seed(而不是 4 个),这样第 3c 步也能真的跑起来(3v3),
# 不至于到最后一步才发现有问题。
SMOKE_TEST = False

# ============================ 窗口 ============================
# 半开区间 [start, stop)。guide 用的是闭区间写法 cv_data[:, w0:w1+1, :],
# 所以 (500, 1501) == guide 的 (500, 1500)。
WINDOWS = {
    "TS":   (500, 1501),   # 1001 帧 —— 过渡区,MLTSA 真正关心的那个问题
    "late": (2000, 2500),  #  500 帧 —— 接近稳态,容易的对照
}
RUN_SENSITIVITY   = False
SENSITIVITY_KEY   = "late2100"
SENSITIVITY_RANGE = (2100, 2500)

# ============================ 模型 ============================
RF_PARAMS = dict(
    n_estimators=300,                    # 正式跑;500 太慢,300 够用
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced_subsample",   # 99 IN / 74 OUT
    n_jobs=-1,
)

N_SPLITS       = 5
SEEDS          = list(range(10))
PERM_REPEATS   = 10      # guide §2 指定 20(v4 里是 10,偏噪)
PERM_N_JOBS    = -1      # permutation 是全流程最贵的一步
TOP_K_JACCARD  = 15      # guide §3:top-15 重叠
MIN_RELIABILITY = 0.20   # 第 3c 步:地板低于这个值就不做衰减校正。
                         # 0.05 太松 —— 实测地板 0.059 时比值算出 0.94,
                         # 而真相是"两窗口完全不同"。近零数相除不可信。
RUN_SHAP       = True

N_PROTEIN_RESIDUES   = 224   # CV_0..223
N_WATERS             = 50    # CV_224..273
N_FEATURES_EXPECTED  = N_PROTEIN_RESIDUES + N_WATERS

# ---------------- 冒烟模式的覆盖 ----------------
if SMOKE_TEST:
    SEEDS = list(range(6))
    N_SPLITS = 3
    PERM_REPEATS = 3
    RF_PARAMS = dict(RF_PARAMS, n_estimators=100)
    RUN_SENSITIVITY = False
    OUT_DIR = OUT_DIR + "_smoke"
    print(">>> 冒烟模式:结果不可用于报告,只用来确认流程能跑通 <<<")

RUN_SHAP = RUN_SHAP and HAVE_SHAP

# ---------------- fail fast ----------------
# v4 的 Configuration cell 也是这么做的:路径不对就当场报错,别跑到一半才发现。
for _lbl, _p in [("H5_PATH", H5_PATH), ("PDB_PATH", PDB_PATH), ("WATERS_PATH", WATERS_PATH)]:
    if not os.path.exists(_p):
        raise FileNotFoundError(f"{_lbl} 不存在:{_p!r} —— 改上面的路径再跑")
os.makedirs(OUT_DIR, exist_ok=True)

ALL_WINDOWS = dict(WINDOWS)
if RUN_SENSITIVITY:
    ALL_WINDOWS[SENSITIVITY_KEY] = SENSITIVITY_RANGE

print(f"输出目录 : {OUT_DIR}")
print(f"窗口     : {ALL_WINDOWS}")
print(f"seed     : {len(SEEDS)} 个  折数: {N_SPLITS}  perm repeats: {PERM_REPEATS}")
print(f"SHAP     : {'开' if RUN_SHAP else '关'}")
print(f"总训练量 : {len(ALL_WINDOWS)} 窗口 × {len(SEEDS)} seed × {N_SPLITS} 折 "
      f"= {len(ALL_WINDOWS) * len(SEEDS) * N_SPLITS} 次 RF 拟合")

输出目录 : /mnt/data1/student/trypsin/yucheng/07.12/v4
窗口     : {'TS': (500, 1501), 'late': (2000, 2500)}
seed     : 10 个  折数: 5  perm repeats: 10
SHAP     : 关
总训练量 : 2 窗口 × 10 seed × 5 折 = 100 次 RF 拟合


## 第 1 步 — 真实特征名

guide §1 + §6 第一条:**最终图里绝对不能再出现 "CV_195"。**

映射规则(来自 `code_PBJ/trypsin_code/rescue_feature_names.ipynb`):

- `CV_0 … CV_223` = 配体到第 i 个蛋白残基的最近距离(拓扑顺序)
- `CV_224 … CV_273` = 配体到第 i 近的水

**水的坑:**"第 i 近的水"每一帧是不同的物理分子,所以只能按 rank 标注,不能当成
固定的 HOH id。这里把 rank 作为显示名,原始 HOH id 只留在单独一列做存档 —— caption
里必须写明这个 caveat(guide §1)。

顺带:跑完这一步回头看 v4 的旧结果就有意思了 —— `CV_230` 是水 rank 6(你晚期窗口
permutation 的第 2 名),`CV_253` 是水 rank 29;而 `CV_195/196/197/198/201` 互相
相关 0.87–0.96 且序列相邻,是一整段 loop。

In [3]:
def build_feature_names():
    """返回 dict:name / kind / resid / raw_water_id,每个都是长度 274 的数组。

    name  绘图用的显示名,例如 'GLY214' 或 'water rank6'
    kind  'protein' / 'water' —— guide 要求两条 track 分开处理、分色
    resid 蛋白残基的 PDB 残基号(Fig 3 的 x 轴);水填 -1
    """
    n = N_FEATURES_EXPECTED
    if not HAVE_MDTRAJ:
        warnings.warn("mdtraj 不可用 —— 退回 CV_i 命名。这样的表只能自己看,"
                      "不能进最终图(guide §6 第一条)。", RuntimeWarning)
        return {"name": np.array([f"CV_{i}" for i in range(n)]),
                "kind": np.array(["unknown"] * n),
                "resid": np.full(n, -1, dtype=int),
                "raw_water_id": np.array([""] * n)}

    top = md.load(PDB_PATH).topology
    residues = [r for r in top.residues if r.is_protein]
    if len(residues) != N_PROTEIN_RESIDUES:
        raise ValueError(
            f"拓扑里有 {len(residues)} 个蛋白残基,期望 {N_PROTEIN_RESIDUES}。"
            "PDB 和 h5 对不上,后面所有残基名都会错位 —— 先查这个。")

    # 用 r.name + r.resSeq 而不是 str(r):str(r) 给的是 'ILE16' 这种拼好的串,
    # 但 Fig 3 的 x 轴需要残基号本身(19–241,中间有 chymotrypsin 编号的空位),
    # 所以两个分开存。
    prot_names  = np.array([f"{r.name}{r.resSeq}" for r in residues])
    prot_resids = np.array([r.resSeq for r in residues], dtype=int)

    waters = np.load(WATERS_PATH, allow_pickle=True)
    if len(waters) != N_WATERS:
        raise ValueError(f"水文件里有 {len(waters)} 条,期望 {N_WATERS}")
    water_ids   = np.array([str(w) for w in waters])
    water_names = np.array([f"water rank{i}" for i in range(N_WATERS)])

    out = {
        "name":  np.concatenate([prot_names, water_names]),
        "kind":  np.array(["protein"] * N_PROTEIN_RESIDUES + ["water"] * N_WATERS),
        "resid": np.concatenate([prot_resids, np.full(N_WATERS, -1, dtype=int)]),
        "raw_water_id": np.concatenate([np.array([""] * N_PROTEIN_RESIDUES), water_ids]),
    }
    assert len(out["name"]) == n, "特征名长度对不上 274"
    return out


names = build_feature_names()
print(f"蛋白残基 {int((names['kind'] == 'protein').sum())} 个,"
      f"水 {int((names['kind'] == 'water').sum())} 个")
print("前 5 个 :", names["name"][:5].tolist())
print("末 3 个 :", names["name"][-3:].tolist())
if HAVE_MDTRAJ:
    r = names["resid"][names["kind"] == "protein"]
    print(f"残基号范围: {r.min()}–{r.max()}   (guide 说的 19–241)")
    # v4 旧结果里那几个 CV 现在叫什么
    for i in (195, 196, 197, 198, 201, 230, 253):
        print(f"  CV_{i:<3d} -> {names['name'][i]}")

蛋白残基 224 个,水 50 个
前 5 个 : ['ILE19', 'VAL20', 'GLY21', 'GLY22', 'TYR23']
末 3 个 : ['water rank47', 'water rank48', 'water rank49']
残基号范围: 1–241   (guide 说的 19–241)
  CV_195 -> GLY214
  CV_196 -> CYS215
  CV_197 -> ALA216
  CV_198 -> GLN217
  CV_201 -> LYS220
  CV_230 -> water rank6
  CV_253 -> water rank29


## 数据加载 + 窗口平均

guide §0:**轨迹级** —— 一条轨迹在窗口内取一个均值向量,不要把单帧当样本。
这正是 Edina 在群里表扬的那一步("that averaging denoises the signal"),
也是 Pedro 审计时点出的、Ayidh 和你最大的差别。

`cv_data` 是 173×2500×274 float32 ≈ 475 MB,而每个窗口平均完只有 173×274 ≈ 190 KB。
所以在同一个 cell 里把所有窗口都切出来再 `del` —— **千万别按窗口分别重启 kernel 跑**,
那样既慢,又很容易让两边的 split 不一致。

In [4]:
t0 = time.time()
with h5py.File(H5_PATH, "r") as h:
    cv_data    = h["cv_data"][:]
    labels_raw = h["labels"][:]
    groups     = h["replica_ids"][:]

labels = np.array([v.decode(errors="replace") if isinstance(v, (bytes, np.bytes_)) else str(v)
                   for v in labels_raw])
# 和 v4 的 load_data 保持同一套编码:IN=0, OUT=1
y = np.array([0 if lab == "IN" else 1 for lab in labels], dtype=np.int64)

if cv_data.shape[2] != N_FEATURES_EXPECTED:
    raise ValueError(f"特征数是 {cv_data.shape[2]},期望 {N_FEATURES_EXPECTED}")
if not np.isfinite(cv_data).all():
    raise ValueError("cv_data 里有 NaN/Inf,先清理")

Xs = {}
for key, (start, stop) in ALL_WINDOWS.items():
    if not (0 <= start < stop <= cv_data.shape[1]):
        raise ValueError(f"窗口 {key}=({start},{stop}) 超出 {cv_data.shape[1]} 帧")
    Xs[key] = cv_data[:, start:stop, :].mean(axis=1).astype(np.float64)
    print(f"  {key:10s} frames [{start}, {stop})  {stop-start:4d} 帧  ->  X {Xs[key].shape}")

del cv_data   # 释放那 475 MB
print(f"\nn_traj={len(y)}  IN={(y==0).sum()}  OUT={(y==1).sum()}  "
      f"unique_groups={len(np.unique(groups))}   ({time.time()-t0:.0f}s)")

  TS         frames [500, 1501)  1001 帧  ->  X (173, 274)
  late       frames [2000, 2500)   500 帧  ->  X (173, 274)

n_traj=173  IN=99  OUT=74  unique_groups=173   (0s)


### split:只能由 seed 决定

和 v4 的 `grouped_splits` 同一套逻辑(`StratifiedGroupKFold`,shuffle,seed)。

**关键:`random_state` 只吃 seed,不掺任何窗口信息。** 这样同一个 seed 在两个窗口下
产生的折**完全相同** —— 这是 guide §0"唯一变量只能是窗口"的技术前提,也是最容易
手滑的地方(比如把窗口 key 混进 `random_state`)。

下面那个 assert 会真的检查一遍,别靠信仰。

In [5]:
def grouped_splits(y, groups, n_splits, seed):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idx = np.arange(len(y))
    return [(tr, te) for tr, te in splitter.split(idx, y, groups)]


# 检查 1:同一 seed 两次调用完全一致
for s in SEEDS:
    a, b = grouped_splits(y, groups, N_SPLITS, s), grouped_splits(y, groups, N_SPLITS, s)
    for (t1, e1), (t2, e2) in zip(a, b):
        assert np.array_equal(t1, t2) and np.array_equal(e1, e2), f"seed={s} split 不可复现"

# 检查 2:不同 seed 确实给出不同的折(否则 seed 平均和第 3b/3c 步都没意义)
s0 = [set(te.tolist()) for _, te in grouped_splits(y, groups, N_SPLITS, SEEDS[0])]
s1 = [set(te.tolist()) for _, te in grouped_splits(y, groups, N_SPLITS, SEEDS[1])]
assert s0 != s1, "不同 seed 给出了相同的折 —— 检查 shuffle=True"

print("split 检查通过:同 seed 可复现,异 seed 有差异。")
print(f"每折测试集大小: {[len(te) for _, te in grouped_splits(y, groups, N_SPLITS, SEEDS[0])]}")

split 检查通过:同 seed 可复现,异 seed 有差异。
每折测试集大小: [35, 35, 35, 34, 34]


## 第 2 步 — 三种重要性方法

guide §2:三种全部在 **held-out 测试行**上算,然后在折和 seed 上平均。

| 方法 | 量的是什么 | 地位 |
|---|---|---|
| **Gini** | 模型内部怎么用这个特征 | 便宜,偏向高基数特征,只当参考 |
| **Permutation** | held-out 上的真实预测贡献 | **结论以它为准** |
| **SHAP** | 每样本归因,取 mean \|SHAP\| | 和 Gini 同源,**不是**独立验证 |

**guide §8 的警告要记牢:**对树模型,Gini 和 TreeExplainer-SHAP 都从同一批 split
推导,量的是同一件事。她 demo 里 Gini↔SHAP 相关到 0.95/0.96,而 permutation 和
两者都只有 0.02。所以 **permutation 才是那根独立的轴,绝不能把 SHAP 说成是对
Gini 的确认。**

### 顺带算一个 guide 里没有的东西:方向

Edina 15:15 那张图按 "ligand farther in OUT (dist ↑)" / "ligand closer in OUT
(dist ↓)" 上色。重要性只说"这个坐标要紧",方向说"要紧在哪一边",信息量大得多。
很便宜,顺手算掉。

这是**描述性标注**,不参与建模、不参与特征选择,所以在全体轨迹上算不构成泄漏 ——
但报告里最好写明这一点。

In [6]:
def compute_shap_mean_abs(model, X_test):
    """mean |SHAP| per feature,取正类(OUT=1)。

    不同 shap 版本返回格式不一样,两种都得接:
      shap <  0.45 : list [ (n,f) class0, (n,f) class1 ]
      shap >= 0.45 : ndarray (n, f, n_classes)
    二分类 RF 下两类的 |SHAP| 数值其实相同(两类和为零),取 class 1 只是
    为了和 guide 的写法一致。
    """
    sv = shap.TreeExplainer(model).shap_values(X_test)
    if isinstance(sv, list):
        arr = sv[1] if len(sv) > 1 else sv[0]
    elif isinstance(sv, np.ndarray) and sv.ndim == 3:
        arr = sv[:, :, 1] if sv.shape[2] > 1 else sv[:, :, 0]
    else:
        arr = sv
    return np.abs(arr).mean(axis=0)


def fi_one_seed(X, y, groups, seed, n_splits=None, run_shap=None):
    """一个窗口 × 一个 seed:跑完 n_splits 折,返回折平均后的三条 FI 向量。

    模型的 random_state 也随 seed 和 fold 变 —— 这是有意的。第 3b/3c 步要量的是
    **整条流水线**的 FI 不稳定性(重采样 + 重训),不只是 split 的抖动;把模型
    seed 固定死会让地板偏乐观。
    """
    n_splits = N_SPLITS if n_splits is None else n_splits
    run_shap = RUN_SHAP if run_shap is None else run_shap

    splits = grouped_splits(y, groups, n_splits, seed)
    nf = X.shape[1]
    acc  = {"gini": np.zeros(nf), "perm": np.zeros(nf), "shap": np.zeros(nf)}
    nok  = {"gini": 0, "perm": 0, "shap": 0}
    oof_pred = np.full(len(y), -1, dtype=int)

    for fold, (tr, te) in enumerate(splits):
        rf = RandomForestClassifier(random_state=10_000 * seed + fold, **RF_PARAMS)
        rf.fit(X[tr], y[tr])
        oof_pred[te] = rf.predict(X[te])

        # (a) Gini —— 来自训练好的树,不需要测试数据
        acc["gini"] += rf.feature_importances_; nok["gini"] += 1

        # 测试折只有单一类别时 balanced accuracy 没意义,跳过 held-out 的两种方法
        if len(np.unique(y[te])) < 2:
            warnings.warn(f"seed={seed} fold={fold} 测试折单一类别,跳过 perm/shap")
            continue

        # (b) permutation —— guide §2 指定 balanced_accuracy
        p = permutation_importance(rf, X[te], y[te], scoring="balanced_accuracy",
                                   n_repeats=PERM_REPEATS,
                                   random_state=20_000 + 100 * seed + fold,
                                   n_jobs=PERM_N_JOBS)
        acc["perm"] += p.importances_mean; nok["perm"] += 1

        # (c) SHAP —— 同样只在 held-out 行上算
        if run_shap:
            try:
                acc["shap"] += compute_shap_mean_abs(rf, X[te]); nok["shap"] += 1
            except Exception as exc:
                warnings.warn(f"SHAP 失败 (seed={seed} fold={fold}): {exc}")

    out = {k: (acc[k] / nok[k] if nok[k] else np.full(nf, np.nan)) for k in acc}
    out["_oof"] = balanced_accuracy_score(y, oof_pred)
    return out


def direction_annotation(X, y):
    """IN→OUT 方向标注:标准化均值差(Cohen's d 量纲),符号即方向。"""
    a, b = X[y == 1], X[y == 0]                      # OUT, IN
    pooled = np.sqrt((a.var(axis=0, ddof=1) + b.var(axis=0, ddof=1)) / 2.0)
    d = (a.mean(axis=0) - b.mean(axis=0)) / (pooled + 1e-12)
    return {"direction_effect": d, "direction_sign": np.sign(d)}


print("函数已定义。下一个 cell 是全流程最贵的一步。")

函数已定义。下一个 cell 是全流程最贵的一步。


### 主循环(最贵的一步)

跑完会**立刻存盘**到 `fi_per_seed.npz`。后面所有分析和画图都从这个文件读,
所以:

- 第 3 步之后的 cell 报错了,不用重跑这一格
- 重启 kernel 之后,跳过这一格,直接跑下面的「从缓存恢复」

这是 notebook 版相对 `.py` 版最实际的好处。

In [7]:
t0 = time.time()
per_seed, fi_mean, oof = {}, {}, {}

for wkey in ALL_WINDOWS:
    stacks = {"gini": [], "perm": [], "shap": []}
    oof[wkey] = []
    for seed in SEEDS:
        ts = time.time()
        r = fi_one_seed(Xs[wkey], y, groups, seed)
        for m in stacks:
            stacks[m].append(r[m])
        oof[wkey].append(r["_oof"])
        print(f"  {wkey:10s} seed={seed:<2d} OOF bal-acc={r['_oof']:.3f}  ({time.time()-ts:.0f}s)")
    per_seed[wkey] = {m: np.vstack(v) for m, v in stacks.items()}
    # seed 平均 —— guide §0 要求的 "average importances over >=5 seeds"
    fi_mean[wkey] = {m: per_seed[wkey][m].mean(axis=0) for m in stacks}
    print(f"  {wkey:10s} 平均 OOF = {np.mean(oof[wkey]):.3f} ± {np.std(oof[wkey]):.3f}\n")

# ---- 立刻存盘 ----
_npz = {f"{w}__{m}": per_seed[w][m] for w in per_seed for m in per_seed[w]}
_npz["seeds"] = np.array(SEEDS)
_npz["feature_names"] = names["name"]
_npz["feature_kind"]  = names["kind"]
_npz["feature_resid"] = names["resid"]
for w in ALL_WINDOWS:
    _npz[f"{w}__oof"]    = np.array(oof[w])
    _npz[f"{w}__window"] = np.array(ALL_WINDOWS[w])
np.savez(os.path.join(OUT_DIR, "fi_per_seed.npz"), **_npz)

print(f"总用时 {time.time()-t0:.0f}s。已存 {OUT_DIR}/fi_per_seed.npz")

# ---- 诊断:permutation importance 的零值 / 并列情况 ----
# 这不是形式主义。274 个特征、每折才 ~35 条测试行,很多特征打乱之后
# balanced accuracy 一点不变 -> permutation importance 恰好等于 0。
# 零值太多会导致两件坏事:
#   1) spearmanr 对近似常数的向量返回 NaN
#   2) top-k 里全是并列,argsort 选谁纯属任意,Jaccard 变成噪声
#
# 但零值最常见的成因不是"特征不重要",而是**特征之间高度相关**:
# 打乱 CV_197 之后,和它相关 0.93 的 CV_198 还在,模型照样预测得准,
# 于是 CV_197 的 permutation importance = 0。这是 permutation importance
# 的已知失效模式,不是 bug。
# 你 v4 的输出里 CV_195/196/197/198/201 互相相关 0.87–0.96、top perm 只有
# 0.0198、大量并列在 0.00429 —— 这个问题在你的数据上已经在发生了。
#
# 所以跑完先看这里,再决定下面几步的数字能不能信。
print("=" * 62)
print("permutation importance 诊断")
for wkey in per_seed:
    M = per_seed[wkey]["perm"]
    zero_frac = float((np.abs(M) < 1e-12).mean())
    v = fi_mean[wkey]["perm"]
    sv = np.sort(v)[::-1]
    tied_at_k = bool(len(sv) > TOP_K_JACCARD and sv[TOP_K_JACCARD - 1] == sv[TOP_K_JACCARD])
    n_distinct = len(np.unique(np.round(v, 12)))
    flag = ""
    if zero_frac > 0.8:
        flag = "  <<< 零值过多,下面的 rho / Jaccard 都不可信"
    elif tied_at_k:
        flag = "  <<< top-k 边界并列,Jaccard 会偏噪"
    print(f"  {wkey:10s} 恰好为 0 的比例={zero_frac:.1%}   "
          f"不同取值={n_distinct}/{len(v)}   top{TOP_K_JACCARD}边界并列={tied_at_k}{flag}")
print("  对策(按推荐顺序):")
print("    1) 把高相关的残基**成组打乱**(block permutation)—— 直接对症,")
print("       比如 201-208 exit loop 当成一个坐标,而不是 8 个互相掩盖的特征;")
print("    2) 提高 PERM_REPEATS / 增加 SEEDS —— 只能缓解噪声,治不了相关性;")
print("    3) 退而用 SHAP 当主方法 —— 它在相关特征上会把贡献摊开而不是归零,")
print("       但要记得 guide §8 的警告:SHAP 和 Gini 同源,不是独立验证。")
print("=" * 62)

  TS         seed=0  OOF bal-acc=0.659  (119s)
  TS         seed=1  OOF bal-acc=0.648  (119s)
  TS         seed=2  OOF bal-acc=0.664  (119s)
  TS         seed=3  OOF bal-acc=0.666  (119s)
  TS         seed=4  OOF bal-acc=0.671  (119s)
  TS         seed=5  OOF bal-acc=0.663  (119s)
  TS         seed=6  OOF bal-acc=0.676  (118s)
  TS         seed=7  OOF bal-acc=0.683  (118s)
  TS         seed=8  OOF bal-acc=0.649  (119s)
  TS         seed=9  OOF bal-acc=0.656  (118s)
  TS         平均 OOF = 0.664 ± 0.011

  late       seed=0  OOF bal-acc=0.749  (117s)
  late       seed=1  OOF bal-acc=0.767  (117s)
  late       seed=2  OOF bal-acc=0.777  (117s)
  late       seed=3  OOF bal-acc=0.772  (117s)
  late       seed=4  OOF bal-acc=0.754  (117s)
  late       seed=5  OOF bal-acc=0.723  (117s)
  late       seed=6  OOF bal-acc=0.757  (117s)
  late       seed=7  OOF bal-acc=0.752  (117s)
  late       seed=8  OOF bal-acc=0.769  (117s)
  late       seed=9  OOF bal-acc=0.784  (118s)
  late       平均 OOF = 0

### 从缓存恢复(可选)

重启 kernel 之后,跑完第 0/1 步的配置 cell,然后**跳过主循环**直接跑这一格。

In [8]:
# 只在需要时执行:从 npz 恢复 per_seed / fi_mean / oof,跳过重训
RESTORE_FROM_CACHE = False

if RESTORE_FROM_CACHE:
    z = np.load(os.path.join(OUT_DIR, "fi_per_seed.npz"), allow_pickle=True)
    wins = sorted({k.split("__")[0] for k in z.files if "__" in k and not k.endswith(("oof", "window"))})
    per_seed = {w: {m: z[f"{w}__{m}"] for m in ("gini", "perm", "shap") if f"{w}__{m}" in z.files}
                for w in wins}
    fi_mean  = {w: {m: per_seed[w][m].mean(axis=0) for m in per_seed[w]} for w in wins}
    oof      = {w: z[f"{w}__oof"].tolist() for w in wins}
    names    = {"name": z["feature_names"], "kind": z["feature_kind"],
                "resid": z["feature_resid"],
                "raw_water_id": np.array([""] * len(z["feature_names"]))}
    print("已从缓存恢复:", wins)
    for w in wins:
        print(f"  {w:10s} 平均 OOF = {np.mean(oof[w]):.3f}")
    print("注意:Xs 没有恢复(方向标注需要它)。要写 CSV 的话把数据加载那格也跑一遍。")
else:
    print("跳过(RESTORE_FROM_CACHE = False)")

跳过(RESTORE_FROM_CACHE = False)


## 第 3 步 — 方法一致性(→ Fig 1)

guide §3:在每个窗口内对三种方法做 rank 相关,看哪些特征是稳健重要、哪些是
某个指标的 artefact。

slide 上想说的那句话:
> "三种方法在 ρ ≈ 0.x 上一致,下面这些特征是三种方法**都**选出来的,
> 所以不是某个指标的 artefact。"

对照 guide §8 她的 demo 结果:

| 方法对 | TS | late |
|---|:---:|:---:|
| Gini ↔ SHAP | 0.95 | 0.96 |
| Gini ↔ Permutation | 0.02 | 0.29 |
| Permutation ↔ SHAP | 0.02 | 0.31 |

如果你的数也是这个形状,就印证了"Gini 和 SHAP 同源、permutation 是独立那根轴"。

In [9]:
def method_agreement(fi, top_k=None):
    """3×3 Spearman 矩阵 + top-k Jaccard。Fig 1 直接吃这个。"""
    top_k = TOP_K_JACCARD if top_k is None else top_k
    methods = [m for m in ("gini", "perm", "shap") if np.isfinite(fi[m]).any()]
    n = len(methods)
    rho = np.full((n, n), np.nan); jac = np.full((n, n), np.nan)
    tops = {m: set(np.argsort(fi[m])[::-1][:top_k]) for m in methods}
    for i, a in enumerate(methods):
        for j, b in enumerate(methods):
            rho[i, j] = spearmanr(fi[a], fi[b]).statistic
            u = len(tops[a] | tops[b])
            jac[i, j] = len(tops[a] & tops[b]) / u if u else np.nan
    return methods, rho, jac


agreement_rows = []
for wkey in per_seed:
    methods, rho, jac = method_agreement(fi_mean[wkey])
    print(f"[{wkey}]")
    for i, a in enumerate(methods):
        for j, b in enumerate(methods):
            agreement_rows.append((wkey, a, b, rho[i, j], jac[i, j]))
            if j > i:
                print(f"   {a:5s} <-> {b:5s}  rho={rho[i,j]:+.3f}   "
                      f"jaccard@{TOP_K_JACCARD}={jac[i,j]:.3f}")
    print()

# 每个窗口 permutation 的 top-10,用真实名字
for wkey in per_seed:
    top = np.argsort(fi_mean[wkey]["perm"])[::-1][:10]
    print(f"[{wkey}] permutation top-10:")
    for r, i in enumerate(top, 1):
        print(f"   {r:2d}. {names['name'][i]:<16s} ({names['kind'][i]:7s})  "
              f"{fi_mean[wkey]['perm'][i]:.5f}")
    print()

[TS]
   gini  <-> perm   rho=+0.108   jaccard@15=0.154

[late]
   gini  <-> perm   rho=-0.230   jaccard@15=0.071

[TS] permutation top-10:
    1. water rank28     (water  )  0.01039
    2. water rank41     (water  )  0.00621
    3. water rank3      (water  )  0.00359
    4. water rank2      (water  )  0.00335
    5. GLY63            (protein)  0.00322
    6. water rank43     (water  )  0.00316
    7. water rank24     (water  )  0.00295
    8. SER145           (protein)  0.00244
    9. ILE174           (protein)  0.00215
   10. water rank6      (water  )  0.00207

[late] permutation top-10:
    1. TYR183           (protein)  0.00141
    2. TYR32            (protein)  0.00085
    3. LEU184           (protein)  0.00076
    4. water rank39     (water  )  0.00067
    5. VAL199           (protein)  0.00060
    6. ASN49            (protein)  0.00033
    7. VAL209           (protein)  0.00030
    8. LYS220           (protein)  0.00028
    9. water rank36     (water  )  0.00026
   10. LYS226   

## 第 3b 步 — 窗口内 FI 复现基线

**这一步 guide 里没有,但不做的话 Fig 2 标题里那个 ρ 没法解释。**

Edina 的 spoiler 是跨窗口 ρ ≈ 0.05。但 Ayidh 15:25 说过 "the top FIs are not the
same for me from one model to another" —— FI 本身就不稳。n=173、每折测试集才 ~35 条,
permutation importance 完全可能跟**任何东西**都不相关。

所以先量出 FI 自己和自己的相关性:同一个窗口、不同 seed 之间的 ρ。这是**噪声地板**。

### 还有一个更要紧的问题:全局 Spearman 本身就靠不住

274 个特征里真正有信号的大概只有 10–20 个,其余 250+ 个在两个窗口下都是噪声,
它们之间的排序每次重跑都是随机洗牌。Spearman 是对全部 274 个名次算的,
**会被这 250+ 个死死压向 0,不管真实结构如何。**

打个比方:274 个学生考试,只有 15 个真复习了,259 个瞎蒙。考两次,**全班名次**的
相关性接近 0(因为 259 个每次都乱序),但"**前 15 名是谁**"是稳定的。

所以下面同时算 `topk_jaccard` 和 `restricted_rho`(只在"至少在一个窗口进过 top-k"
的特征子集上重算 ρ)。**结论看这两个,不看全局 ρ。**

In [10]:
def topk_overlap(a, b, k=None):
    """两条 FI 向量的 top-k 重叠 —— 比全局 Spearman 可靠得多。"""
    k = TOP_K_JACCARD if k is None else k
    # 并列告警:permutation importance 常有大量恰好相等的值(尤其是 0)。
    # 一旦第 k 名和第 k+1 名数值相同,argsort 选谁进 top-k 就是任意的,
    # Jaccard 会变成噪声。这里只标记,不改行为 —— 诊断在主循环那一格。
    sa, sb = np.sort(a)[::-1], np.sort(b)[::-1]
    tied = (len(a) > k and sa[k - 1] == sa[k]) or (len(b) > k and sb[k - 1] == sb[k])
    ta = set(np.argsort(a)[::-1][:k]); tb = set(np.argsort(b)[::-1][:k])
    u = np.array(sorted(ta | tb))
    return {"k": k, "jaccard": len(ta & tb) / len(ta | tb), "n_shared": len(ta & tb),
            "restricted_rho": float(spearmanr(a[u], b[u]).statistic),
            "boundary_tied": bool(tied)}


def reproducibility_baseline(per_seed, method="perm", top_k=None):
    """窗口内 vs 跨窗口,全部用**单 seed vs 单 seed**,噪声量级匹配。"""
    top_k = TOP_K_JACCARD if top_k is None else top_k
    wins = list(per_seed)
    res = {"method": method, "windows": wins, "top_k": top_k}

    for w in wins:
        M = per_seed[w][method]
        pr = list(combinations(range(len(M)), 2))
        rs = [spearmanr(M[i], M[j]).statistic for i, j in pr]
        ov = [topk_overlap(M[i], M[j], top_k) for i, j in pr]
        res[f"within_{w}"] = {
            "pairs": len(rs), "rho_mean": float(np.mean(rs)),
            "rho_min": float(np.min(rs)), "rho_max": float(np.max(rs)),
            "topk_jaccard_mean": float(np.mean([o["jaccard"] for o in ov])),
            "topk_restricted_rho_mean": float(np.mean([o["restricted_rho"] for o in ov]))}

    for wa, wb in combinations(wins, 2):
        A, B = per_seed[wa][method], per_seed[wb][method]
        ij = [(i, j) for i in range(len(A)) for j in range(len(B))]
        rs = [spearmanr(A[i], B[j]).statistic for i, j in ij]
        ov = [topk_overlap(A[i], B[j], top_k) for i, j in ij]
        res[f"cross_{wa}_vs_{wb}"] = {
            "pairs": len(rs), "rho_mean": float(np.mean(rs)),
            "rho_min": float(np.min(rs)), "rho_max": float(np.max(rs)),
            "topk_jaccard_mean": float(np.mean([o["jaccard"] for o in ov])),
            "topk_restricted_rho_mean": float(np.mean([o["restricted_rho"] for o in ov]))}
    return res


repro = reproducibility_baseline(per_seed, "perm")

print("=== 全局 rho(仅供参考,会被 250+ 个噪声特征拽向 0)===")
for k, v in repro.items():
    if k.startswith("within_"):
        print(f"  窗口内 {k[7:]:10s} rho={v['rho_mean']:+.3f} "
              f"[{v['rho_min']:+.3f}, {v['rho_max']:+.3f}]   <- 噪声地板")
for k, v in repro.items():
    if k.startswith("cross_"):
        print(f"  跨窗口 {k[6:]:22s} rho={v['rho_mean']:+.3f} "
              f"[{v['rho_min']:+.3f}, {v['rho_max']:+.3f}]")

print(f"\n=== top-{TOP_K_JACCARD} 重叠(结论看这个)===")
for k, v in repro.items():
    if k.startswith("within_"):
        print(f"  窗口内 {k[7:]:10s} Jaccard={v['topk_jaccard_mean']:.3f}  "
              f"restricted_rho={v['topk_restricted_rho_mean']:+.3f}   <- 噪声地板")
for k, v in repro.items():
    if k.startswith("cross_"):
        print(f"  跨窗口 {k[6:]:22s} Jaccard={v['topk_jaccard_mean']:.3f}  "
              f"restricted_rho={v['topk_restricted_rho_mean']:+.3f}")
print("\n判据:窗口内 Jaccard 明显高于跨窗口 -> 重要特征集合确实换了。")
print("      窗口内 Jaccard 本身就低      -> FI 连自己都复现不了,先别下结论。")

=== 全局 rho(仅供参考,会被 250+ 个噪声特征拽向 0)===
  窗口内 TS         rho=+0.024 [-0.236, +0.187]   <- 噪声地板
  窗口内 late       rho=+0.034 [-0.145, +0.220]   <- 噪声地板
  跨窗口 TS_vs_late             rho=-0.015 [-0.193, +0.161]

=== top-15 重叠(结论看这个)===
  窗口内 TS         Jaccard=0.057  restricted_rho=-0.586   <- 噪声地板
  窗口内 late       Jaccard=0.054  restricted_rho=-0.572   <- 噪声地板
  跨窗口 TS_vs_late             Jaccard=0.033  restricted_rho=-0.665

判据:窗口内 Jaccard 明显高于跨窗口 -> 重要特征集合确实换了。
      窗口内 Jaccard 本身就低      -> FI 连自己都复现不了,先别下结论。


## 第 3c 步 — split-half 地板 + 衰减校正

第 3b 步的地板用的是**单 seed vs 单 seed**,而 Fig 2 的 headline 用的是
**10-seed 平均**后的两条向量。后者干净得多,两者不可比 —— 拿单 seed 的地板去衬
seed 平均的 headline,会人为地让"两个窗口不同"这个结论看起来更强。

**做法:把 10 个 seed 随机对半分成 A、B,每半各自平均。**

```
地板      ρ(TS_A, TS_B)       同窗口,两边都是 5-seed 平均
          ρ(late_A, late_B)
headline  ρ(TS_A, late_B)     跨窗口,同样 5v5,而且 seed 集合不相交
          ρ(TS_B, late_A)     —— 不会因为共享噪声而虚高
```

枚举多种对半方式再取中位数,降低地板本身的估计方差。

### 衰减校正(Spearman 1904)

$$\rho_{\text{true}} = \frac{\rho_{\text{cross}}}{\sqrt{\rho_{\text{TS}} \cdot \rho_{\text{late}}}}$$

| 校正后 | 含义 |
|---|---|
| ≈ 1 | 两个窗口其实用**同一批**坐标,观测到的低 ρ 全是噪声 → guide §8 的 headline 不成立 |
| ≈ 0 | 真的是不同坐标 → 结论成立,而且有了误差标尺 |
| 0.3–0.7 | 部分重叠 → 别说"完全不相关",改成报告哪些共有、哪些特有 |
| 分母 < 0.05 | FI 复现性太差,这个问题用现有数据答不了 |

**为什么必须做这一步:**在合成数据上(答案已知)测过,"两窗口用相同特征"和
"用不同特征"两种情形的观测 ρ 分别是 +0.15 和 −0.03 —— 都"接近 0",光看这个数字
分不开,而真相完全相反。除以地板之后才各归其位(+0.87 vs −0.11)。

### 一个限制,要写进报告

seed 变的是 CV 划分和 RF 的 `random_state`,**不包括**重新采样那 173 条轨迹
(数据集是固定的)。所以这个地板量的是"给定这批轨迹,FI 有多可复现",不是
"换一批轨迹还能不能复现"。后者只会更低,所以这里的校正其实偏保守。

In [11]:
def split_half_floor(per_seed, method="perm", top_k=None, max_partitions=200, rng_seed=0):
    """降噪程度匹配的地板 + 衰减校正。需要偶数个、至少 6 个 seed。"""
    top_k = TOP_K_JACCARD if top_k is None else top_k
    wins = list(per_seed)
    n_seeds = per_seed[wins[0]][method].shape[0]
    if n_seeds < 6 or n_seeds % 2:
        warnings.warn(f"split-half 需要偶数个、至少 6 个 seed,当前 {n_seeds} 个。"
                      "把 SEEDS 设成 10 个再跑。", RuntimeWarning)
        return {"error": f"n_seeds={n_seeds} 不满足要求"}

    half = n_seeds // 2
    parts = [set(c) for c in combinations(range(n_seeds), half)]
    parts = parts[:len(parts) // 2]          # (A,B) 与 (B,A) 等价,去重
    if len(parts) > max_partitions:
        rng = np.random.default_rng(rng_seed)
        parts = [parts[i] for i in rng.choice(len(parts), max_partitions, replace=False)]

    res = {"method": method, "n_seeds": n_seeds, "half": half,
           "n_partitions": len(parts), "top_k": top_k}

    floors, floor_series = {}, {}
    for w in wins:
        M = per_seed[w][method]
        rs, js = [], []
        for A in parts:
            B = [i for i in range(n_seeds) if i not in A]
            a, b = M[sorted(A)].mean(axis=0), M[B].mean(axis=0)
            rs.append(spearmanr(a, b).statistic)
            js.append(topk_overlap(a, b, top_k)["jaccard"])
        floors[w] = float(np.median(rs))
        floor_series[w] = np.array(rs)          # 逐 partition 保留,下面配对用
        res[f"floor_{w}"] = {"rho_median": float(np.median(rs)),
                             "rho_p05": float(np.percentile(rs, 5)),
                             "rho_p95": float(np.percentile(rs, 95)),
                             "topk_jaccard_median": float(np.median(js))}

    for wa, wb in combinations(wins, 2):
        Ma, Mb = per_seed[wa][method], per_seed[wb][method]
        rs, js = [], []
        for A in parts:
            B = [i for i in range(n_seeds) if i not in A]
            a1, a2 = Ma[sorted(A)].mean(axis=0), Ma[B].mean(axis=0)
            b1, b2 = Mb[sorted(A)].mean(axis=0), Mb[B].mean(axis=0)
            rs.append(np.mean([spearmanr(a1, b2).statistic, spearmanr(a2, b1).statistic]))
            js.append(np.mean([topk_overlap(a1, b2, top_k)["jaccard"],
                               topk_overlap(a2, b1, top_k)["jaccard"]]))
        rho_cross = float(np.median(rs))
        # 逐 partition 配对算校正值 —— 点估计相除会掩盖不确定性。
        # 地板 0.059 / 观测 0.064 这种情形,比值是 0.94,但两个数都是噪声,
        # 结论完全不可信。给区间才看得出来。
        cross_series = np.array(rs)
        fa, fb = floor_series[wa], floor_series[wb]
        with np.errstate(invalid="ignore", divide="ignore"):
            dis_series = cross_series / np.sqrt(np.where(fa * fb > 0, fa * fb, np.nan))
        dis_series = dis_series[np.isfinite(dis_series)]

        e = {"rho_median": rho_cross,
             "rho_p05": float(np.percentile(rs, 5)), "rho_p95": float(np.percentile(rs, 95)),
             "topk_jaccard_median": float(np.median(js)),
             "floor_rho": [floors[wa], floors[wb]]}

        # ---- 判定 ----
        # MIN_RELIABILITY 是地板的最低可信值。0.05 太松:测试发现地板 0.059、
        # 观测 0.064 时比值算出 0.94,而真相是"两窗口完全不同" —— 两个近零
        # 噪声数相除会给出自信的错误答案。0.20 是个保守的下限。
        jac_within = np.mean([res[f"floor_{wa}"]["topk_jaccard_median"],
                              res[f"floor_{wb}"]["topk_jaccard_median"]])
        jac_cross = e["topk_jaccard_median"]
        e["jaccard_within_mean"] = float(jac_within)

        if not np.isfinite([floors[wa], floors[wb], rho_cross]).all():
            e["disattenuated_rho"] = None
            e["verdict"] = ("rho 是 NaN —— 某个窗口的 FI 向量几乎是常数"
                            "(permutation importance 大面积恰好为 0)。"
                            "先看主循环那格的零值/相关性诊断。")
        elif min(floors[wa], floors[wb]) < MIN_RELIABILITY:
            e["disattenuated_rho"] = None
            e["verdict"] = (
                f"地板太低(rho_within = {floors[wa]:.3f} / {floors[wb]:.3f},"
                f"门槛 {MIN_RELIABILITY})—— 衰减校正在这里不可用:两个近零的噪声数"
                f"相除会给出自信的错误答案。\n"
                f"      改看 top-{top_k} Jaccard:窗口内 {jac_within:.3f} vs "
                f"跨窗口 {jac_cross:.3f}。"
                + ("窗口内明显更高 -> 重要特征集合确实换了(但请以 Jaccard 为准,别报 rho)。"
                   if jac_within > 2 * max(jac_cross, 1e-9)
                   else "两者接近 -> 现有数据答不了这个问题,先做 block permutation 或按 loop 聚合。"))
        else:
            dis = float(np.median(dis_series))
            lo, hi = (float(np.percentile(dis_series, 5)),
                      float(np.percentile(dis_series, 95))) if len(dis_series) else (np.nan, np.nan)
            e["disattenuated_rho"] = dis
            e["disattenuated_ci90"] = [lo, hi]
            wide = not np.isfinite([lo, hi]).all() or (hi - lo) > 0.8
            # 交叉验证:Jaccard 和校正后的 rho 应该讲同一个故事,不一致就别下结论
            jac_says_different = jac_within > 2 * max(jac_cross, 1e-9)
            rho_says_same = dis > 0.7
            if wide:
                e["verdict"] = (f"校正后 rho = {dis:.2f},但 90% 区间 [{lo:.2f}, {hi:.2f}] 太宽 —— "
                                f"不可判定。改看 Jaccard:窗口内 {jac_within:.3f} vs 跨窗口 {jac_cross:.3f}。")
            elif jac_says_different and rho_says_same:
                e["verdict"] = (f"⚠ 两个指标打架:校正后 rho = {dis:.2f} 说'相同',"
                                f"但 Jaccard(窗口内 {jac_within:.3f} vs 跨窗口 {jac_cross:.3f})说'不同'。"
                                "以 Jaccard 为准 —— 地板偏低时 rho 的比值不可靠。")
            elif dis > 0.7:
                e["verdict"] = (f"校正后 rho = {dis:.2f} [{lo:.2f}, {hi:.2f}] —— 两个窗口其实用的是"
                                "同一批坐标,观测到的低 rho 主要是噪声。guide §8 的 headline 在你的数据上不成立。")
            elif dis < 0.3:
                e["verdict"] = (f"校正后 rho = {dis:.2f} [{lo:.2f}, {hi:.2f}] —— 扣掉噪声之后重要特征"
                                "集合仍然基本不重叠。结论成立,而且现在有了误差标尺。")
            else:
                e["verdict"] = (f"校正后 rho = {dis:.2f} [{lo:.2f}, {hi:.2f}] —— 部分重叠。"
                                "别说'完全不相关',改成报告哪些特征共有、哪些各自特有。")
        res[f"cross_{wa}_vs_{wb}"] = e
    return res


shf = split_half_floor(per_seed, "perm")
if "error" in shf:
    print(shf["error"])
else:
    print(f"(基于 {shf['n_partitions']} 种对半方式,每半 {shf['half']} 个 seed)\n")
    for k, v in shf.items():
        if k.startswith("floor_"):
            print(f"  地板 {k[6:]:10s} rho={v['rho_median']:+.3f} "
                  f"[{v['rho_p05']:+.3f}, {v['rho_p95']:+.3f}]   "
                  f"Jaccard={v['topk_jaccard_median']:.3f}")
    print()
    for k, v in shf.items():
        if k.startswith("cross_"):
            d = v["disattenuated_rho"]
            print(f"  跨窗口 {k[6:]}")
            print(f"    观测 rho = {v['rho_median']:+.3f}   "
                  f"Jaccard = {v['topk_jaccard_median']:.3f}")
            print(f"    校正后   = {'N/A' if d is None else format(d, '+.3f')}")
            print(f"    >>> {v['verdict']}\n")

(基于 126 种对半方式,每半 5 个 seed)

  地板 TS         rho=+0.116 [-0.004, +0.193]   Jaccard=0.200
  地板 late       rho=+0.128 [+0.034, +0.224]   Jaccard=0.034

  跨窗口 TS_vs_late
    观测 rho = -0.075   Jaccard = 0.000
    校正后   = N/A
    >>> 地板太低(rho_within = 0.116 / 0.128,门槛 0.2)—— 衰减校正在这里不可用:两个近零的噪声数相除会给出自信的错误答案。
      改看 top-15 Jaccard:窗口内 0.117 vs 跨窗口 0.000。窗口内明显更高 -> 重要特征集合确实换了(但请以 Jaccard 为准,别报 rho)。



## 落盘

产出:

| 文件 | 内容 |
|---|---|
| `fi_table_<window>.csv` | 274 行 × [name, kind, resid, gini, perm, shap, direction] |
| `fi_per_seed.npz` | 每个 (窗口, 方法) 的 (n_seeds, 274) 原始矩阵 |
| `method_agreement.csv` | 每窗口的 3×3 Spearman + top-15 Jaccard → Fig 1 |
| `reproducibility.json` | 第 3b + 3c 步的全部数字和 verdict |
| `feature_names.csv` | 274 个特征名对照表,存档 |

**第 4 步(四张图 + PyMOL)全部从 `fi_table_*.csv` 和 `fi_per_seed.npz` 读,
不用再重训任何模型。**

In [12]:
import csv

# 每窗口一张 274 行的表
for wkey, (start, stop) in ALL_WINDOWS.items():
    if wkey not in fi_mean:
        continue
    d = direction_annotation(Xs[wkey], y)
    p = os.path.join(OUT_DIR, f"fi_table_{wkey}.csv")
    with open(p, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["index", "name", "kind", "resid", "raw_water_id",
                    "gini", "perm", "shap", "direction_effect", "direction_sign"])
        for i in range(len(names["name"])):
            w.writerow([i, names["name"][i], names["kind"][i], names["resid"][i],
                        names["raw_water_id"][i],
                        f"{fi_mean[wkey]['gini'][i]:.8g}",
                        f"{fi_mean[wkey]['perm'][i]:.8g}",
                        f"{fi_mean[wkey]['shap'][i]:.8g}",
                        f"{d['direction_effect'][i]:.6g}",
                        int(d["direction_sign"][i])])
    print(f"  {p}   (frames [{start},{stop}))")

# 方法一致性
p = os.path.join(OUT_DIR, "method_agreement.csv")
with open(p, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["window", "method_a", "method_b", "spearman_rho", f"top{TOP_K_JACCARD}_jaccard"])
    for row in agreement_rows:
        w.writerow([row[0], row[1], row[2], f"{row[3]:.4f}", f"{row[4]:.4f}"])
print(f"  {p}")

# 特征名对照
p = os.path.join(OUT_DIR, "feature_names.csv")
with open(p, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["index", "name", "kind", "resid", "raw_water_id"])
    for i in range(len(names["name"])):
        w.writerow([i, names["name"][i], names["kind"][i],
                    names["resid"][i], names["raw_water_id"][i]])
print(f"  {p}")

# 复现性 + 配置
payload = {
    "reproducibility_3b": repro,
    "split_half_floor_3c": shf,
    "oof_balanced_accuracy": {w: {"per_seed": list(map(float, oof[w])),
                                  "mean": float(np.mean(oof[w]))} for w in oof},
    "config": {"windows": {k: list(v) for k, v in ALL_WINDOWS.items()},
               "rf_params": {k: v for k, v in RF_PARAMS.items()},
               "n_splits": N_SPLITS, "seeds": SEEDS,
               "perm_repeats": PERM_REPEATS, "top_k": TOP_K_JACCARD,
               "smoke_test": SMOKE_TEST, "run_shap": RUN_SHAP},
}
p = os.path.join(OUT_DIR, "reproducibility.json")
with open(p, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2, ensure_ascii=False)
print(f"  {p}")

if SMOKE_TEST:
    print("\n>>> 这是冒烟模式的结果,不能用于报告。"
          "把 SMOKE_TEST 改成 False,重启 kernel 从头跑一遍。 <<<")

  /mnt/data1/student/trypsin/yucheng/07.12/v4/fi_table_TS.csv   (frames [500,1501))
  /mnt/data1/student/trypsin/yucheng/07.12/v4/fi_table_late.csv   (frames [2000,2500))
  /mnt/data1/student/trypsin/yucheng/07.12/v4/method_agreement.csv
  /mnt/data1/student/trypsin/yucheng/07.12/v4/feature_names.csv
  /mnt/data1/student/trypsin/yucheng/07.12/v4/reproducibility.json


---

## 跑完先看什么

**1. 第 3c 步的地板。** 如果 `floor_TS` / `floor_late` 的 rho 低于 0.05,脚本会
直接告诉你这个问题现在答不了 —— 那就先加 seed、提高 `PERM_REPEATS`,或者按 loop
把残基聚合成组再比。

**2. 校正后的 rho 和它的 verdict。** 这一句可以直接进报告。

**3. 每个窗口 permutation 的 top-10。** 对照 guide §8 她的 demo:

- TS 窗口她拿到的是**水占多数**(rank41/18/2/28/34/24)+ GLY214、ASN95、TYR40、PHE179、SER62
- late 窗口是 **201–208 loop**(CYS201、GLY207、ILE208、GLN206)+ SER126、THR225

你 v4 的旧结果里 `CV_195/196/197/198/201` 是一整段相关到 0.87–0.96 的连续 loop,
`CV_230` 是水 rank 6 —— 现在有了真名,可以直接看是不是同一批。

## 下一步(第 4 步)

从 `fi_table_*.csv` 和 `fi_per_seed.npz` 出四张图:

1. **Fig 1** 三方法一致性热图 + top-15 Jaccard
2. **Fig 2** 跨窗口 FI 散点,画 y=x,标 top-10 名字,蛋白/水分色,标题给 ρ(配 3c 的地板)
3. **Fig 3** 沿序列 profile,TS 上 / late 下,共享 y 轴,50 个水单独小 panel,按 direction 着色
4. **Fig 4** PyMOL 双图:per-residue FI 写进 B-factor,两窗口同色标同取向

不需要重训任何模型。